In [6]:
import os
import time
from rfsoc_radio.overlay import RadioOverlay

current_dir = os.path.abspath("")
bit_path = os.path.join(current_dir, "replay_rfsoc_radio.bit")

ol = RadioOverlay(bitfile_name=bit_path, run_test=True, debug_test=False)

print("Detected IPs:", list(ol.ip_dict.keys()))

Running BPSK Synchronisation Test...
 BPSK Synchronisation Successful! ✔  

Running QPSK Synchronisation Test...
 QPSK Synchronisation Successful! ✔  

Detected IPs: ['usp_rf_data_converter', 'replay_buf_top_0', 'axi_dma_tx', 'axi_intc', 'axi_dma_rx', 'DataInspectorRx/axi_dma', 'receiver', 'transmitter', 'DataInspectorRx/data_inspector_module', 'DataInspectorTx/axi_dma', 'DataInspectorTx/data_inspector_module', 'zynq_ultra_ps_e']


In [2]:
import json

# PYNQ parses the HWH into ip_dict — check what it found for your IP
import pprint
pprint.pprint(ol.ip_dict['replay_buf_top_0'])

{'addr_range': 4096,
 'bdtype': None,
 'device': <pynq.pl_server.embedded_device.EmbeddedDevice object at 0xffff95e39ab0>,
 'driver': <class 'pynq.overlay.DefaultIP'>,
 'fullpath': 'replay_buf_top_0',
 'gpio': {},
 'interrupts': {},
 'mem_id': 's_axi',
 'memtype': 'REGISTER',
 'parameters': {'ADDR_WIDTH': '5',
                'ARUSER_WIDTH': '0',
                'AWUSER_WIDTH': '0',
                'BRAM_ADDR_WIDTH': '16',
                'BUSER_WIDTH': '0',
                'CLK_DOMAIN': 'rfsoc_radio_usp_rf_data_converter_0_clk_dac2',
                'C_BASEADDR': '0xA0040000',
                'C_HIGHADDR': '0xA0040FFF',
                'Component_Name': 'rfsoc_radio_replay_buf_top_0_0',
                'DATA_WIDTH': '32',
                'EDK_IPTYPE': 'PERIPHERAL',
                'FORWARD_TLAST': 'true',
                'FREQ_HZ': '128000000',
                'HAS_BRESP': '1',
                'HAS_BURST': '0',
                'HAS_CACHE': '0',
                'HAS_LOCK': '0',
       

In [7]:
from pynq import MMIO

# Base address from the TCL patch: 0xA00E0000
# Or read it from ip_dict:
base_addr = ol.ip_dict['replay_buf_top_0']['phys_addr']
print(f"Base address: 0x{base_addr:08X}")

rb_mmio = MMIO(base_addr, length=0x20)

# Register offsets from replay_ctrl_axi.vhd:
CTRL      = 0x00
REP_COUNT = 0x04
STATUS    = 0x08
REP_DONE  = 0x0C

# Write / read wrappers
def rb_write(offset, value):
    rb_mmio.write(offset, value)

def rb_read(offset):
    return rb_mmio.read(offset)

# Test it
print(f"CTRL      = 0x{rb_read(CTRL):08X}")
print(f"REP_COUNT = 0x{rb_read(REP_COUNT):08X}")
print(f"STATUS    = 0x{rb_read(STATUS):08X}")
print(f"REP_DONE  = 0x{rb_read(REP_DONE):08X}")

Base address: 0xA0040000
CTRL      = 0x00000000
REP_COUNT = 0x00000000
STATUS    = 0x00000000
REP_DONE  = 0x00000000


In [8]:
ol.radio_receiver.terminal()

Accordion(children=(HBox(children=(VBox(children=(Textarea(value='Received data will appear here...\r', disabl…

In [9]:
rb_write(CTRL, 0x0)   # bit0=mode=0, ensure passthrough
ol.radio_transmitter.data('Hello World!\r')
ol.radio_transmitter.start()
time.sleep(1.0)


In [10]:
ol.radio_transmitter.start()
rb_write(REP_COUNT, 10)    # REP_COUNT
rb_write(CTRL, 0x01)  # replay mode=1, finite=bit1=0 

# buffer is filled continously, so single transmition of tx works here
while not (rb_read(STATUS) & 0x02):
    time.sleep(0.05)

print(f"Replay done. Reps: {rb_read(REP_DONE)}")
rb_write(CTRL, 0x0) # pass through
ol.radio_transmitter.stop()

Replay done. Reps: 10


In [11]:
ol.radio_transmitter.start()
rb_write(REP_COUNT, 0)    # rep cout
rb_write(CTRL, 0x03)   # mode=1, infinite=1
print("Infinite replay running.")

time.sleep(2)
rb_mmio.write(0x00, 0x0)
print(f"\nStopped. reps_done={rb_read(REP_DONE)}")

ol.radio_transmitter.stop()

Infinite replay running.

Stopped. reps_done=1580
